# Diagnóstico del modelo — pedidos tardíos

Siete diagnósticos antes de modificar variables o partición.
No altera el modelo ni los datos.

1. Tasa de atraso por semana (serie temporal + cortes de partición)
2. Censura: pedidos sin entrega al cierre de datos
3. Fechas exactas de los tres conjuntos
4. Oráculo temporal (semana de compra como única variable)
5. GBT con las mismas variables vs. logística
6. Validación hacia adelante mensual de la logística
7. Ablación: grupos ordenados por impacto en log-loss

In [1]:
import os, sys

sys.path.insert(0, '/home/notebooks')
for _p in ['/repo/etapa2/src', '/repo/etapa2/lib/sparkstats/src']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

# Misma configuración que el notebook principal
JARS = ','.join([
    '/opt/spark/extra-jars/iceberg-spark-runtime-4.1_2.13-1.11.0.jar',
    '/opt/spark/extra-jars/hadoop-aws-3.4.2.jar',
    '/opt/spark/extra-jars/aws-sdk-bundle-v2-2.29.52.jar',
    '/opt/spark/extra-jars/postgresql-42.7.4.jar',
])

spark = (
    SparkSession.builder
    .appName('diagnostico_modelo')
    .master('spark://spark-master:7077')
    .config('spark.jars', JARS)
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.lake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lake.type', 'hadoop')
    .config('spark.sql.catalog.lake.warehouse', 's3a://olist/warehouse')
    .config('spark.hadoop.fs.s3a.endpoint', os.environ['RUSTFS_ENDPOINT'])
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .config('spark.executor.memory', '3g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

26/10/22 12:19:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 4.1.3


In [2]:
# Cargar df_features desde checkpoint
_CHECKPOINT_PATH = 's3a://olist/checkpoints/df_features'
df = spark.read.parquet(_CHECKPOINT_PATH).cache()
print(f'Filas totales: {df.count():,}')
print('Columnas:', df.columns[:10], '...')

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/22 12:19:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Filas totales: 94,797
Columnas: ['order_id', 'seller_id_critico', 'customer_zip_code_prefix', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'] ...


In [3]:
# Constantes de partición (deben coincidir con config.py)
FECHA_FIN_TRAIN = '2017-12-31'
FECHA_FIN_VAL   = '2018-03-31'
FECHA_FIN_TEST  = '2018-08-31'

df_train = df.filter(F.col('fecha_compra') <= FECHA_FIN_TRAIN)
df_val   = df.filter((F.col('fecha_compra') > FECHA_FIN_TRAIN) & (F.col('fecha_compra') <= FECHA_FIN_VAL))
df_test  = df.filter((F.col('fecha_compra') > FECHA_FIN_VAL)  & (F.col('fecha_compra') <= FECHA_FIN_TEST))

print(f'Train: {df_train.count():,}  |  Val: {df_val.count():,}  |  Test: {df_test.count():,}')

Train: 43,293  |  Val: 20,589  |  Test: 30,915


## D1 · Tasa de atraso por semana de compra

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import os
import pandas as pd

DIR_DIAG = '../mediciones/etapa3/diagnostico'
os.makedirs(DIR_DIAG, exist_ok=True)

semanal = (
    df
    .withColumn('semana', F.date_trunc('week', 'fecha_compra').cast('date'))
    .groupBy('semana')
    .agg(
        F.count('*').alias('n'),
        F.mean(F.col('tarde').cast(DoubleType())).alias('tasa_tardios'),
    )
    .orderBy('semana')
    .toPandas()
)
semanal['semana'] = pd.to_datetime(semanal['semana'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=semanal['semana'], y=semanal['tasa_tardios'],
                         mode='lines+markers', name='Tasa tardíos', line=dict(color='steelblue')))

# add_vline con fecha como string falla en esta versión de Plotly (sum(strings) → int+str).
# Solución: convertir a ms desde epoch, que es la representación interna de Plotly para datetimes.
for fecha, label, color in [
    (FECHA_FIN_TRAIN, 'fin train', 'green'),
    (FECHA_FIN_VAL,   'fin val',   'orange'),
    (FECHA_FIN_TEST,  'fin test',  'red'),
]:
    x_ms = int(pd.Timestamp(fecha).value // 1_000_000)
    fig.add_vline(x=x_ms, line_dash='dash', line_color=color,
                  annotation_text=label, annotation_position='top right')

fig.update_layout(
    title='Tasa de atraso por semana de compra',
    xaxis_title='Semana', yaxis_title='Tasa tardíos',
    yaxis_tickformat='.1%',
    height=420,
)
pio.write_image(fig, f'{DIR_DIAG}/D1_tasa_semanal.png', width=1000, height=420)
fig.show()
print(semanal[['semana', 'n', 'tasa_tardios']].to_string(index=False))

In [ ]:
# Tasa por mes dentro del período de prueba
mensual_test = (
    df_test
    .withColumn('mes_compra', F.date_format('fecha_compra', 'yyyy-MM'))
    .groupBy('mes_compra')
    .agg(
        F.count('*').alias('n'),
        F.mean(F.col('tarde').cast(DoubleType())).alias('tasa_tardios'),
    )
    .orderBy('mes_compra')
    .toPandas()
)
print('Tasa de tardíos por mes en PRUEBA:')
print(mensual_test.to_string(index=False))

## D2 · Censura: pedidos sin entrega al cierre de datos

In [ ]:
# Leer silver.pedidos para incluir pedidos no entregados
pedidos_raw = spark.read.table('lake.silver.pedidos')

# Fecha de cierre = máxima fecha de compra registrada
cierre = pedidos_raw.agg(F.max('fecha_compra').alias('cierre')).collect()[0]['cierre']
print(f'Fecha de cierre estimada: {cierre}')

censura = (
    pedidos_raw
    .filter(F.col('fecha_compra').between('2018-01-01', '2018-08-31'))
    .withColumn('mes_compra', F.date_format('fecha_compra', 'yyyy-MM'))
    .groupBy('mes_compra')
    .agg(
        F.count('*').alias('n_total'),
        F.sum(F.when(F.col('order_delivered_customer_date').isNull(), 1).otherwise(0)).alias('sin_entrega'),
        F.sum(
            F.when(
                F.col('order_delivered_customer_date').isNull() &
                F.col('order_estimated_delivery_date').isNotNull() &
                (F.col('order_estimated_delivery_date') > F.lit(cierre)),
                1
            ).otherwise(0)
        ).alias('prometida_post_cierre'),
    )
    .withColumn('pct_sin_entrega', F.col('sin_entrega') / F.col('n_total'))
    .orderBy('mes_compra')
    .toPandas()
)
print('Censura por mes de compra (2018):')
print(censura.to_string(index=False))

## D3 · Fechas exactas de los tres conjuntos

In [ ]:
for nombre, dset in [('TRAIN', df_train), ('VAL', df_val), ('TEST', df_test)]:
    row = dset.agg(
        F.min('fecha_compra').alias('primer_dia'),
        F.max('fecha_compra').alias('ultimo_dia'),
        F.count('*').alias('n'),
        F.mean(F.col('tarde').cast(DoubleType())).alias('prevalencia'),
    ).collect()[0]
    print(f'{nombre}: {row["primer_dia"]} → {row["ultimo_dia"]}  '
          f'| n={row["n"]:,}  | tardíos={row["prevalencia"]:.2%}')

# Verificar ausencia de solapamiento
solapamiento = (
    df_val.filter(F.col('fecha_compra') <= FECHA_FIN_TRAIN).count() +
    df_test.filter(F.col('fecha_compra') <= FECHA_FIN_VAL).count()
)
print(f'\nPedidos en solapamiento (deben ser 0): {solapamiento}')

## D4 · Oráculo temporal: semana de compra como única variable

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import sys
for _p in ['/repo/etapa2/src', '/repo/etapa2/lib/sparkstats/src']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Oráculo: semana ISO como string (yyyyWww evita confusión entre año y semana)
df_oraculo_train = (
    df_train
    .withColumn('semana_str',
        F.concat(F.year('fecha_compra').cast('string'),
                 F.lit('W'),
                 F.lpad(F.weekofyear('fecha_compra').cast('string'), 2, '0')))
    .select('semana_str', F.col('tarde').cast('int').alias('tarde'))
)
df_oraculo_test = (
    df_test
    .withColumn('semana_str',
        F.concat(F.year('fecha_compra').cast('string'),
                 F.lit('W'),
                 F.lpad(F.weekofyear('fecha_compra').cast('string'), 2, '0')))
    .select('semana_str', F.col('tarde').cast('int').alias('tarde'))
)

idx = StringIndexer(inputCol='semana_str', outputCol='semana_idx', handleInvalid='keep')
enc = OneHotEncoder(inputCol='semana_idx', outputCol='semana_vec', handleInvalid='keep')
asm = VectorAssembler(inputCols=['semana_vec'], outputCol='features')
lr_or = LogisticRegression(featuresCol='features', labelCol='tarde',
                            predictionCol='prediction', rawPredictionCol='rawPrediction',
                            probabilityCol='probability',
                            regParam=0.01, maxIter=100)
pipe_oraculo = Pipeline(stages=[idx, enc, asm, lr_or])
modelo_oraculo = pipe_oraculo.fit(df_oraculo_train)

evaluator_roc = BinaryClassificationEvaluator(
    labelCol='tarde', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

pred_train_or = modelo_oraculo.transform(df_oraculo_train)
pred_test_or  = modelo_oraculo.transform(df_oraculo_test)

roc_oraculo_train = evaluator_roc.evaluate(pred_train_or)
roc_oraculo_test  = evaluator_roc.evaluate(pred_test_or)

print('Oráculo temporal (semana de compra):')
print(f'  ROC-AUC entrenamiento : {roc_oraculo_train:.4f}')
print(f'  ROC-AUC prueba        : {roc_oraculo_test:.4f}')
print(f'  (ROC-AUC modelo logístico en prueba: 0.6626)')
if roc_oraculo_train > 0.70:
    print('\nALERTA: el oráculo explica mucho del target → hay señal temporal fuerte.')
else:
    print('\nEl oráculo no supera 0.70 en entrenamiento → la señal temporal es moderada.')

## D5 · GBT con las mismas variables vs. logística

In [ ]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.functions import vector_to_array
from sparkstats.evaluacion import log_loss as calc_log_loss
from modelo.transformadores import construir_pipeline

df_train_int = df_train.withColumn('tarde', F.col('tarde').cast('int'))
df_val_int   = df_val.withColumn('tarde',   F.col('tarde').cast('int'))
df_test_int  = df_test.withColumn('tarde',  F.col('tarde').cast('int'))

# ── Entrenar la logística con los hiperparámetros elegidos ──────────────────
print('Entrenando logística (reg=0.036, elastic=0.025, min_ev=30)...')
pipe_lr = construir_pipeline(reg_param=0.036, elastic_net_param=0.025, min_eventos=30)
modelo_lr = pipe_lr.fit(df_train_int)
print('Logística entrenada.')

# ── Preprocesador compartido (todo menos el último estimador = LR) ──────────
from pyspark.ml import PipelineModel
preprocesador = PipelineModel(stages=modelo_lr.stages[:-1])

df_train_prep = preprocesador.transform(df_train_int)
df_val_prep   = preprocesador.transform(df_val_int)
df_test_prep  = preprocesador.transform(df_test_int)

# ── GBT con los mismos features ─────────────────────────────────────────────
# Spark 4.x: GBTClassifier no acepta probabilityCol/rawPredictionCol en el constructor;
# los nombres de columna de salida ('probability', 'rawPrediction') son fijos por defecto.
print('Entrenando GBT (maxIter=50, maxDepth=5)...')
gbt = GBTClassifier(
    featuresCol='features', labelCol='tarde',
    predictionCol='prediction',
    maxIter=50, maxDepth=5, stepSize=0.1, subsamplingRate=0.8, seed=42,
)
modelo_gbt = gbt.fit(df_train_prep)
print('GBT entrenado.')

# ── Evaluación comparativa ──────────────────────────────────────────────────
ev_roc = BinaryClassificationEvaluator(labelCol='tarde', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
ev_pr  = BinaryClassificationEvaluator(labelCol='tarde', rawPredictionCol='rawPrediction', metricName='areaUnderPR')

print()
print(f'{"Modelo":<15} {"Conjunto":<12} {"ROC-AUC":>8} {"PR-AUC":>8} {"log-loss":>10}')
print('-' * 57)

for modelo_nombre, modelo_obj, usa_prep in [
    ('Logística', modelo_lr,  False),
    ('GBT',       modelo_gbt, True),
]:
    for conj_nombre, df_int, df_prep in [
        ('Validación', df_val_int,  df_val_prep),
        ('Prueba',     df_test_int, df_test_prep),
    ]:
        src = df_prep if usa_prep else df_int
        pred = modelo_obj.transform(src)
        pred = pred.withColumn('prob_1', vector_to_array(F.col('probability'))[1])
        roc = ev_roc.evaluate(pred)
        pr  = ev_pr.evaluate(pred)
        ll  = calc_log_loss(pred.withColumn('tarde', F.col('tarde').cast(DoubleType())), 'tarde', 'prob_1')
        print(f'{modelo_nombre:<15} {conj_nombre:<12} {roc:>8.4f} {pr:>8.4f} {ll:>10.4f}')

## D6 · Validación hacia adelante mensual de la logística

In [ ]:
from modelo.transformadores import construir_pipeline
from pyspark.ml.functions import vector_to_array
import pandas as pd

meses_2018 = ['2018-01', '2018-02', '2018-03', '2018-04',
              '2018-05', '2018-06', '2018-07', '2018-08']
resultados_wf = []

ev_roc_wf = BinaryClassificationEvaluator(
    labelCol='tarde', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
ev_pr_wf  = BinaryClassificationEvaluator(
    labelCol='tarde', rawPredictionCol='rawPrediction', metricName='areaUnderPR')

for mes in meses_2018:
    anio, m = mes.split('-')
    m_int = int(m)
    corte_inicio = f'{anio}-{m}-01'
    corte_fin    = f'{anio}-{m_int+1:02d}-01' if m_int < 12 else f'{int(anio)+1}-01-01'

    df_mes_eval = df.filter(
        (F.col('fecha_compra') >= corte_inicio) & (F.col('fecha_compra') < corte_fin)
    )
    df_antes = df.filter(F.col('fecha_compra') < corte_inicio)

    n_eval = df_mes_eval.count()
    if n_eval < 100:
        print(f'{mes}: solo {n_eval} pedidos, saltando.')
        continue

    df_antes_int = df_antes.withColumn('tarde', F.col('tarde').cast('int'))
    df_mes_int   = df_mes_eval.withColumn('tarde', F.col('tarde').cast('int'))

    pipe_wf   = construir_pipeline(reg_param=0.036, elastic_net_param=0.025, min_eventos=30)
    modelo_wf = pipe_wf.fit(df_antes_int)
    pred_wf   = modelo_wf.transform(df_mes_int)
    pred_wf   = pred_wf.withColumn('prob_1', vector_to_array(F.col('probability'))[1])

    roc      = ev_roc_wf.evaluate(pred_wf)
    pr       = ev_pr_wf.evaluate(pred_wf)
    prev_obs  = df_mes_int.agg(F.mean(F.col('tarde').cast(DoubleType()))).collect()[0][0]
    prev_pred = pred_wf.agg(F.mean(F.col('prob_1'))).collect()[0][0]

    fila = dict(mes=mes, n=n_eval,
                roc_auc=round(roc, 4), pr_auc=round(pr, 4),
                prevalencia_obs=round(prev_obs, 4), prevalencia_pred=round(prev_pred, 4))
    resultados_wf.append(fila)
    print(f'{mes}: n={n_eval:,}  ROC={roc:.4f}  PR={pr:.4f}'
          f'  prev_obs={prev_obs:.3f}  prev_pred={prev_pred:.3f}')

df_wf = pd.DataFrame(resultados_wf)
print('\nValidación hacia adelante mensual — resumen:')
print(df_wf.to_string(index=False))
df_wf.to_csv(f'{DIR_DIAG}/D6_walk_forward_mensual.csv', index=False)
print(f'Guardado en {DIR_DIAG}/D6_walk_forward_mensual.csv')

In [ ]:
# Gráfico D6
fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=df_wf['mes'], y=df_wf['roc_auc'], mode='lines+markers', name='ROC-AUC'))
fig6.add_trace(go.Scatter(x=df_wf['mes'], y=df_wf['pr_auc'],  mode='lines+markers', name='PR-AUC'))
fig6.add_trace(go.Scatter(x=df_wf['mes'], y=df_wf['prevalencia_obs'],  mode='lines+markers', name='Prevalencia obs.', line=dict(dash='dot')))
fig6.add_trace(go.Scatter(x=df_wf['mes'], y=df_wf['prevalencia_pred'], mode='lines+markers', name='Prevalencia pred.', line=dict(dash='dash')))
fig6.update_layout(title='Validación hacia adelante mensual — logística', height=420)
pio.write_image(fig6, f'{DIR_DIAG}/D6_walk_forward.png', width=900, height=420)
fig6.show()

## D7 · Ablación: grupos ordenados por impacto en log-loss

In [ ]:
import glob, pandas as pd

# Buscar el CSV de ablación generado en el notebook principal
# El notebook principal guarda relativo a su CWD (etapa3/), así que buscar en ambos lugares
patron_opciones = [
    'mediciones/etapa3/logistica/ablacion*.csv',
    '../mediciones/etapa3/logistica/ablacion*.csv',
    '/home/notebooks/etapa3/mediciones/etapa3/logistica/ablacion*.csv',
]
archivos = []
for patron in patron_opciones:
    archivos = sorted(glob.glob(patron))
    if archivos:
        break

print(f'Archivos de ablación encontrados: {archivos}')

if archivos:
    df_abl = pd.read_csv(archivos[-1])
    df_abl = df_abl.sort_values('delta_log_loss', ascending=False)
    print('\nAblación — grupos ordenados por impacto (mayor delta = más importante):')
    print(df_abl.to_string(index=False))

    fig7 = go.Figure(go.Bar(
        x=df_abl['grupo'], y=df_abl['delta_log_loss'],
        text=[f'{v:.4f}' for v in df_abl['delta_log_loss']],
        textposition='outside',
    ))
    fig7.update_layout(
        title='Ablación: Δ log-loss al quitar cada grupo (mayor = más importante)',
        xaxis_title='Grupo', yaxis_title='Δ log-loss',
        height=400,
    )
    pio.write_image(fig7, f'{DIR_DIAG}/D7_ablacion.png', width=900, height=400)
    fig7.show()
else:
    print('No se encontró el CSV de ablación. Verificar que el paso 10 del notebook principal se ejecutó.')

In [ ]:
spark.stop()
print('Spark detenido. Diagnósticos completados.')